# Notebook 5: Building a Knowledge Graph with the OpenAI Batch API

This notebook does the same thing as Notebook 3 (extract entities from PDFs), but uses the **OpenAI Batch API** instead of real-time API calls.

**Why Batch API?**
- **50% cheaper** — half the cost per token
- **Higher rate limits** — process thousands of chunks without throttling
- **Trade-off:** Results arrive within a 24-hour window (usually much faster)

## How It Works

```
PDF → PyMuPDF text extraction → Split into chunks → Generate prompts
    → Write JSONL file → Upload to OpenAI → Submit batch → Poll for completion
    → Parse results → Write to Neo4j → Entity resolution
```

Instead of calling the API once per chunk in real-time, we generate **all prompts up front**, submit them as a single batch, and process the results when they're ready.

In [ ]:
# You will need to install tesseract via homebrew 
#   sudo apt-get install tesseract-ocr (Linux)
#   brew install tesseract (Mac + Homebrew)
#   On Windows you can download the installer from here: https://github.com/UB-Mannheim/tesseract/wiki or use chocolatey

#Alternatively you can drop tesseract and just use Docling as the only fallback
%pip install -r requirements.txt

## Step 1: Setup

In [1]:
import os
import json
import asyncio
import time
import tempfile
from pathlib import Path

import neo4j
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

driver = neo4j.GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)
driver.verify_connectivity()

client = OpenAI(api_key=OPENAI_API_KEY)

MODEL = "gpt-4.1-2025-04-14"

print("Connected to Neo4j!")
print(f"Using model: {MODEL}")

Connected to Neo4j!
Using model: gpt-4.1-2025-04-14


## Step 2: PDF Text Extraction

Same PyMuPDF extraction as Notebook 3 — with OCR fallback for scanned documents.

In [2]:
import pymupdf
import shutil
import subprocess

data_dir = Path("../data")

if shutil.which("tesseract"):
    print(f"Tesseract found: {shutil.which('tesseract')}")
else:
    print("Tesseract not found (OCR fallback will be skipped).")

pdf_files = sorted(data_dir.glob("*.pdf"))
print(f"\nFound {len(pdf_files)} PDF(s):")
for pdf_file in pdf_files:
    doc = pymupdf.open(pdf_file)
    print(f"  {pdf_file.name}: {len(doc)} pages")
    doc.close()

Tesseract found: /opt/homebrew/bin/tesseract

Found 2 PDF(s):
  attention_is_all_you_need.pdf: 15 pages
  bert_paper.pdf: 16 pages


In [3]:
def extract_page_text(page):
    """Extract text from a single page with OCR fallback."""
    text = page.get_text()
    if text.strip():
        return text, "text"
    try:
        tp = page.get_textpage_ocr(tessdata=None, language="eng", dpi=300)
        ocr_text = page.get_text(textpage=tp)
        if ocr_text.strip():
            return ocr_text, "ocr"
    except Exception:
        pass
    try:
        pix = page.get_pixmap(dpi=300)
        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
            tmpname = f.name
        pix.save(tmpname)
        result = subprocess.run(
            ["tesseract", tmpname, "stdout", "--psm", "3", "-l", "eng"],
            capture_output=True, text=True, timeout=60,
        )
        os.unlink(tmpname)
        if result.returncode == 0 and result.stdout.strip():
            return result.stdout, "ocr"
    except Exception:
        pass
    return "", "empty"


def extract_text_from_pdf(pdf_path):
    """Extract text from a PDF using PyMuPDF with OCR fallback."""
    doc = pymupdf.open(pdf_path)
    pages = []
    stats = {"text": 0, "ocr": 0, "empty": 0, "total": len(doc)}
    for page in doc:
        page_text, method = extract_page_text(page)
        pages.append(page_text)
        stats[method] += 1
    doc.close()
    return "\n\n".join(pages), stats

print("Extraction functions ready.")

Extraction functions ready.


## Step 3: Extract Text & Split into Chunks

We extract text from all PDFs, then split into chunks using the same splitter the pipeline uses.

In [4]:
from neo4j_graphrag.experimental.components.text_splitters.fixed_size_splitter import FixedSizeSplitter

splitter = FixedSizeSplitter(chunk_size=4000, chunk_overlap=200)

# Extract text from all PDFs and split into chunks
all_chunks = []  # list of (pdf_name, chunk_index, chunk_text)

for pdf_path in sorted(data_dir.glob("*.pdf")):
    text, stats = extract_text_from_pdf(pdf_path)
    print(f"{pdf_path.name}: {stats['total']} pages, {len(text):,} chars "
          f"(text: {stats['text']}, ocr: {stats['ocr']}, empty: {stats['empty']})")

    if not text.strip():
        print("  Skipping (no text extracted).")
        continue

    text_chunks = await splitter.run(text)
    for i, chunk in enumerate(text_chunks.chunks):
        all_chunks.append((pdf_path.name, i, chunk.text))

print(f"\nTotal chunks across all PDFs: {len(all_chunks)}")

attention_is_all_you_need.pdf: 15 pages, 39,526 chars (text: 15, ocr: 0, empty: 0)
bert_paper.pdf: 16 pages, 64,147 chars (text: 16, ocr: 0, empty: 0)

Total chunks across all PDFs: 28


## Step 4: Define Schema

Same schema toggle as the other notebooks. For the Batch API, we need the schema as a dict to embed in each prompt.

In [5]:
# ── Toggle: set to False to let the LLM generate its own schema ──
# NOTE: With the Batch API, auto-schema requires an extra synchronous call
# to extract the schema before batching. Manual schema is recommended.
USE_MANUAL_SCHEMA = False

NODE_TYPES = [
    "Person",
    {
        "label": "Organization",
        "description": "A company, university, or research lab",
        "properties": [{"name": "name", "type": "STRING"}],
    },
    {
        "label": "Model",
        "description": "A machine learning model or architecture",
        "properties": [{"name": "name", "type": "STRING"}],
    },
    {
        "label": "Technique",
        "description": "A method, algorithm, or technique",
        "properties": [{"name": "name", "type": "STRING"}],
    },
    {
        "label": "Dataset",
        "description": "A dataset or benchmark used for evaluation",
        "properties": [{"name": "name", "type": "STRING"}],
    },
    {
        "label": "Metric",
        "description": "An evaluation metric (e.g., BLEU, perplexity)",
        "properties": [{"name": "name", "type": "STRING"}],
    },
]

RELATIONSHIP_TYPES = [
    "AUTHORED_BY",
    "AFFILIATED_WITH",
    "USES_TECHNIQUE",
    "EVALUATED_ON",
    "ACHIEVES_SCORE_ON",
    "COMPARED_TO",
    {"label": "IMPROVES_UPON", "description": "One model/technique outperforms another"},
]

PATTERNS = [
    ("Model", "AUTHORED_BY", "Person"),
    ("Person", "AFFILIATED_WITH", "Organization"),
    ("Model", "USES_TECHNIQUE", "Technique"),
    ("Model", "EVALUATED_ON", "Dataset"),
    ("Model", "ACHIEVES_SCORE_ON", "Metric"),
    ("Model", "COMPARED_TO", "Model"),
    ("Model", "IMPROVES_UPON", "Model"),
]

if USE_MANUAL_SCHEMA:
    schema_dict = {
        "node_types": NODE_TYPES,
        "relationship_types": RELATIONSHIP_TYPES,
        "patterns": PATTERNS,
    }
    print(f"Using MANUAL schema: {len(NODE_TYPES)} node types, {len(RELATIONSHIP_TYPES)} relationship types")
else:
    # Auto-schema: use a single synchronous call to extract schema from a sample
    sample_text = all_chunks[0][2] if all_chunks else ""
    print("Using AUTO schema: extracting from a text sample...")
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": (
            "Analyze the following text and propose a schema for a knowledge graph. "
            "Return JSON with: node_types (list of labels), relationship_types (list of labels), "
            "and patterns (list of [source_type, relationship, target_type] triples).\n\n"
            f"Text:\n{sample_text[:3000]}"
        )}],
    )
    schema_dict = json.loads(response.choices[0].message.content)
    print(f"Extracted schema: {json.dumps(schema_dict, indent=2)}")

Using AUTO schema: extracting from a text sample...
Extracted schema: {
  "node_types": [
    "Paper",
    "Author",
    "Organization",
    "Email",
    "Model",
    "Task",
    "Dataset",
    "Metric",
    "Conference",
    "Location",
    "Date",
    "Contribution"
  ],
  "relationship_types": [
    "AUTHORED_BY",
    "AFFILIATED_WITH",
    "HAS_EMAIL",
    "PRESENTED_AT",
    "HELD_IN",
    "HELD_ON",
    "PROPOSES_MODEL",
    "EVALUATED_ON",
    "ACHIEVES_METRIC",
    "PERMISSION_GRANTED_BY",
    "PERMISSION_FOR",
    "CONTRIBUTED_TO",
    "WORK_PERFORMED_AT",
    "IMPROVES_OVER",
    "APPLIES_TO",
    "ESTABLISHES_STATE_OF_THE_ART"
  ],
  "patterns": [
    [
      "Paper",
      "AUTHORED_BY",
      "Author"
    ],
    [
      "Author",
      "AFFILIATED_WITH",
      "Organization"
    ],
    [
      "Author",
      "HAS_EMAIL",
      "Email"
    ],
    [
      "Paper",
      "PRESENTED_AT",
      "Conference"
    ],
    [
      "Conference",
      "HELD_IN",
      "Location"
   

## Step 5: Generate Batch JSONL

We generate the same extraction prompt the `neo4j-graphrag` pipeline uses, then write one request per chunk to a JSONL file for the Batch API.

In [6]:
from neo4j_graphrag.generation.prompts import ERExtractionTemplate

prompt_template = ERExtractionTemplate()

# Generate JSONL file for the Batch API
batch_file = Path("../batch_requests.jsonl")

with open(batch_file, "w") as f:
    for idx, (pdf_name, chunk_idx, chunk_text) in enumerate(all_chunks):
        prompt = prompt_template.format(
            text=chunk_text,
            schema=schema_dict,
            examples="",
        )
        request = {
            "custom_id": f"{pdf_name}__chunk_{chunk_idx}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": MODEL,
                "temperature": 0,
                "response_format": {"type": "json_object"},
                "messages": [{"role": "user", "content": prompt}],
            },
        }
        f.write(json.dumps(request) + "\n")

file_size_mb = batch_file.stat().st_size / (1024 * 1024)
print(f"Created {batch_file.name}: {len(all_chunks)} requests, {file_size_mb:.1f} MB")

Created batch_requests.jsonl: 28 requests, 0.2 MB


## Step 6: Submit Batch

Upload the JSONL file and submit the batch job. The Batch API has a 24-hour completion window, but typically finishes much faster.

In [7]:
# Upload the JSONL file
with open(batch_file, "rb") as f:
    uploaded = client.files.create(file=f, purpose="batch")
print(f"Uploaded file: {uploaded.id}")

# Submit the batch
batch = client.batches.create(
    input_file_id=uploaded.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
)
print(f"Batch submitted: {batch.id}")
print(f"Status: {batch.status}")

Uploaded file: file-TxhYMgo5D5dYAuyYDAumwa
Batch submitted: batch_69a0bce0799c81908b2db601e62e113a
Status: validating


## Step 7: Poll for Completion

Wait for the batch to finish. Depending on the day this could take under an hour, or 24 hours. For large datasets, it's so much faster.

This is set up to use OpenAI -- you can switch to a different platform, you'll just need to change everything to work with the platform's idiosyncracies. 

In [8]:
POLL_INTERVAL = 30  # seconds between status checks

print(f"Waiting for batch {batch.id}...")
while True:
    status = client.batches.retrieve(batch.id)

    completed = status.request_counts.completed
    failed = status.request_counts.failed
    total = status.request_counts.total
    print(f"  [{status.status}] {completed}/{total} completed, {failed} failed")

    if status.status == "completed":
        print(f"\nBatch completed!")
        break
    elif status.status in ("failed", "expired", "cancelled"):
        print(f"\nBatch {status.status}.")
        if status.errors:
            for error in status.errors.data:
                print(f"  Error: {error.message}")
        break

    time.sleep(POLL_INTERVAL)

Waiting for batch batch_69a0bce0799c81908b2db601e62e113a...
  [validating] 0/0 completed, 0 failed
  [validating] 0/0 completed, 0 failed
  [validating] 0/0 completed, 0 failed
  [in_progress] 0/28 completed, 0 failed
  [in_progress] 0/28 completed, 0 failed
  [in_progress] 0/28 completed, 0 failed
  [in_progress] 0/28 completed, 0 failed
  [in_progress] 0/28 completed, 0 failed
  [in_progress] 0/28 completed, 0 failed
  [in_progress] 24/28 completed, 0 failed
  [in_progress] 24/28 completed, 0 failed
  [in_progress] 24/28 completed, 0 failed
  [in_progress] 24/28 completed, 0 failed
  [completed] 28/28 completed, 0 failed

Batch completed!


## Step 8: Parse Results & Write to Neo4j

Download the results, parse each response into entities and relationships, and write them to Neo4j using the same writer the pipeline uses.

In [9]:
from neo4j_graphrag.experimental.components.types import Neo4jNode, Neo4jRelationship, Neo4jGraph
from neo4j_graphrag.experimental.components.kg_writer import Neo4jWriter

# Download results
result_content = client.files.content(status.output_file_id)
result_lines = result_content.text.strip().split("\n")
print(f"Downloaded {len(result_lines)} results.")

# Parse results into Neo4jGraph objects
writer = Neo4jWriter(driver=driver, neo4j_database=NEO4J_DATABASE)

total_nodes = 0
total_rels = 0
errors = 0

for line in result_lines:
    result_obj = json.loads(line)
    custom_id = result_obj["custom_id"]
    response_body = result_obj.get("response", {}).get("body", {})

    # Check for API-level errors
    if result_obj.get("error"):
        print(f"  API error for {custom_id}: {result_obj['error']}")
        errors += 1
        continue

    try:
        content = response_body["choices"][0]["message"]["content"]
        data = json.loads(content)

        # Strip None property values (same fix as notebook 03)
        for node in data.get("nodes", []):
            if "properties" in node:
                node["properties"] = {
                    k: v for k, v in node["properties"].items() if v is not None
                }
        for rel in data.get("relationships", []):
            if "properties" in rel:
                rel["properties"] = {
                    k: v for k, v in rel["properties"].items() if v is not None
                }

        graph = Neo4jGraph.model_validate(data)
        total_nodes += len(graph.nodes)
        total_rels += len(graph.relationships)

        # Write to Neo4j
        await writer.run(graph=graph)

    except Exception as e:
        print(f"  Parse error for {custom_id}: {e}")
        errors += 1

print(f"\nWritten to Neo4j: {total_nodes} nodes, {total_rels} relationships")
if errors:
    print(f"Errors: {errors} chunks failed")

Downloaded 28 results.
  Parse error for bert_paper.pdf__chunk_4: Unterminated string starting at: line 1063 column 72 (char 95224)

Written to Neo4j: 965 nodes, 1069 relationships
Errors: 1 chunks failed


## Step 9: Entity Resolution

In [10]:
from neo4j_graphrag.experimental.components.resolver import (
    SinglePropertyExactMatchResolver,
)

resolver = SinglePropertyExactMatchResolver(driver, neo4j_database=NEO4J_DATABASE)
result = await resolver.run()
print(f"Resolved {result.number_of_nodes_to_resolve} entities \u2192 {result.number_of_created_nodes} merged nodes")

Resolved 965 entities → 584 merged nodes


## Step 10: Explore the Graph

In [11]:
INTERNAL_LABELS = ["__KGBuilder__", "__Entity__", "Chunk", "Document"]

records, _, _ = driver.execute_query(
    """
    MATCH (n:__Entity__)
    WITH [l IN labels(n) WHERE NOT l IN $internal][0] AS entity_type, count(n) AS count
    RETURN entity_type, count
    ORDER BY count DESC
    """,
    {"internal": INTERNAL_LABELS},
    database_=NEO4J_DATABASE,
)

print("=== Entity Types ===")
total = 0
for record in records:
    print(f"  {record['entity_type']}: {record['count']}")
    total += record['count']
print(f"\n  Total entities: {total}")

=== Entity Types ===
  Author: 327
  Paper: 111
  Model: 83
  Task: 60
  Conference: 39
  Dataset: 32
  Date: 27
  Metric: 20
  Email: 12
  Organization: 11
  Contribution: 8
  Location: 4

  Total entities: 734


In [12]:
records, _, _ = driver.execute_query(
    """
    MATCH (a:__Entity__)-[r]->(b:__Entity__)
    RETURN [l IN labels(a) WHERE NOT l IN $internal][0] AS from_type, a.name AS from_name,
           type(r) AS relationship,
           [l IN labels(b) WHERE NOT l IN $internal][0] AS to_type, b.name AS to_name
    ORDER BY from_type, from_name
    LIMIT 30
    """,
    {"internal": INTERNAL_LABELS},
    database_=NEO4J_DATABASE,
)

print(f"=== Knowledge Graph Triples (first 30) ===")
for record in records:
    print(
        f"  ({record['from_type']}: {record['from_name']}) "
        f"-[{record['relationship']}]-> "
        f"({record['to_type']}: {record['to_name']})"
    )

=== Knowledge Graph Triples (first 30) ===
  (Author: Aidan N. Gomez) -[AFFILIATED_WITH]-> (Organization: University of Toronto)
  (Author: Aidan N. Gomez) -[HAS_EMAIL]-> (Email: None)
  (Author: Ashish Vaswani) -[AFFILIATED_WITH]-> (Organization: Google Brain)
  (Author: Ashish Vaswani) -[HAS_EMAIL]-> (Email: None)
  (Author: Clark) -[CONTRIBUTED_TO]-> (Contribution: None)
  (Author: Illia Polosukhin) -[HAS_EMAIL]-> (Email: None)
  (Author: Jacob Devlin) -[AFFILIATED_WITH]-> (Organization: Google AI Language)
  (Author: Jacob Devlin) -[HAS_EMAIL]-> (Email: None)
  (Author: Jakob Uszkoreit) -[AFFILIATED_WITH]-> (Organization: Google Research)
  (Author: Jakob Uszkoreit) -[HAS_EMAIL]-> (Email: None)
  (Author: Kenton Lee) -[AFFILIATED_WITH]-> (Organization: Google AI Language)
  (Author: Kenton Lee) -[HAS_EMAIL]-> (Email: None)
  (Author: Kristina Toutanova) -[AFFILIATED_WITH]-> (Organization: Google AI Language)
  (Author: Kristina Toutanova) -[HAS_EMAIL]-> (Email: None)
  (Author: Lli

## Cleanup

In [13]:
# Clean up the local JSONL file
if batch_file.exists():
    batch_file.unlink()
    print(f"Deleted {batch_file.name}")

driver.close()
print("Done!")

Deleted batch_requests.jsonl
Done!


## Summary

In this notebook you learned how to:
- Generate extraction prompts for all chunks up front
- Submit them as an OpenAI Batch API job (50% cheaper)
- Poll for completion and parse the results
- Write the extracted graph to Neo4j

**When to use Batch API vs real-time (Notebook 3):**

| | Real-time (Notebook 3) | Batch API (this notebook) |
|---|---|---|
| **Speed** | Results in minutes | Results in minutes to hours |
| **Cost** | Full price | 50% cheaper |
| **Rate limits** | Standard limits | Higher limits |
| **Best for** | Interactive exploration, small datasets | Large document sets, scheduled jobs |